# Session 04 · Advanced Pandas Techniques and Operations

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

## Sample data: restaurant orders

In [1]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(4)
n = 60
orders = pd.DataFrame({
    "order_id": range(1, n + 1),
    "date": pd.date_range("2026-10-01", periods=n, freq="8h").strftime("%Y-%m-%d"),
    "branch": rng.choice(["Clifton", "Gulshan", "DHA"], n),
    "channel": rng.choice(["dine-in", "delivery", "takeaway"], n),
    "amount": rng.integers(800, 4000, n),
})
print(orders.head())

   order_id        date   branch   channel  amount
0         1  2026-10-01      DHA   dine-in    2827
1         2  2026-10-01      DHA  delivery    2399
2         3  2026-10-01      DHA  delivery    1105
3         4  2026-10-02  Gulshan  delivery    3859
4         5  2026-10-02      DHA  takeaway    3545


## Example 1 - groupby with named aggregations

In [2]:
summary = orders.groupby("branch").agg(
    orders=("order_id", "count"), revenue=("amount", "sum"), avg_bill=("amount", "mean")).round(0)
print(summary)
assert summary["orders"].sum() == n

         orders  revenue  avg_bill
branch                            
Clifton      13    27966    2151.0
DHA          23    50218    2183.0
Gulshan      24    60707    2529.0


## Example 2 - transform

In [3]:
orders["branch_avg"] = orders.groupby("branch")["amount"].transform("mean")
orders["above_avg"] = orders["amount"] > orders["branch_avg"]
print(orders[["branch", "amount", "branch_avg", "above_avg"]].head())

    branch  amount   branch_avg  above_avg
0      DHA    2827  2183.391304       True
1      DHA    2399  2183.391304       True
2      DHA    1105  2183.391304      False
3  Gulshan    3859  2529.458333       True
4      DHA    3545  2183.391304       True


## Example 3 - pivot table and crosstab

In [4]:
pt = pd.pivot_table(orders, index="branch", columns="channel", values="amount", aggfunc="sum", fill_value=0)
ct = pd.crosstab(orders["branch"], orders["channel"])
print(pt, ct, sep="\n\n")
assert ct.values.sum() == n

channel  delivery  dine-in  takeaway
branch                              
Clifton     11450    11163      5353
DHA         19431    11372     19415
Gulshan     10909    32254     17544

channel  delivery  dine-in  takeaway
branch                              
Clifton         6        4         3
DHA            10        5         8
Gulshan         4       12         8


## Example 4 - merge

In [5]:
branches = pd.DataFrame({"branch": ["Clifton", "Gulshan", "DHA"], "manager": ["Faraz", "Hina", "Usman"]})
full = orders.merge(branches, on="branch", how="left", validate="many_to_one")
print(full[["order_id", "branch", "manager"]].head())
assert len(full) == len(orders)

   order_id   branch manager
0         1      DHA   Usman
1         2      DHA   Usman
2         3      DHA   Usman
3         4  Gulshan    Hina
4         5      DHA   Usman


## Example 5 - melt and pivot

In [6]:
wide = pd.DataFrame({"student": ["Ali", "Sana"], "maths": [72, 91], "physics": [65, 88]})
long = wide.melt(id_vars="student", var_name="subject", value_name="marks")
back = long.pivot(index="student", columns="subject", values="marks")
print(long, back, sep="\n\n")

  student  subject  marks
0     Ali    maths     72
1    Sana    maths     91
2     Ali  physics     65
3    Sana  physics     88

subject  maths  physics
student                
Ali         72       65
Sana        91       88


## Example 6 - dates, resample, rolling

In [7]:
orders["date"] = pd.to_datetime(orders["date"])
daily = orders.set_index("date")["amount"].resample("D").sum()
print(daily.head())
print(daily.resample("W").sum())
print(daily.rolling(3).mean().round(0).head())

date
2026-10-01     6331
2026-10-02    11094
2026-10-03     5411
2026-10-04     5978
2026-10-05     5681
Freq: D, Name: amount, dtype: int64
date
2026-10-04    28814
2026-10-11    46124
2026-10-18    53243
2026-10-25    10710
Freq: W-SUN, Name: amount, dtype: int64
date
2026-10-01       NaN
2026-10-02       NaN
2026-10-03    7612.0
2026-10-04    7494.0
2026-10-05    5690.0
Freq: D, Name: amount, dtype: float64
